# Orekit Validation

In [1]:
import orekit
orekit.initVM()

from orekit.pyhelpers import download_orekit_data_curdir, setup_orekit_curdir
download_orekit_data_curdir()
setup_orekit_curdir()

## Time Conversions

In [2]:
from org.orekit.time import TimeScalesFactory, AbsoluteDate

utc = TimeScalesFactory.getUTC()
tai = TimeScalesFactory.getTAI()
gps = TimeScalesFactory.getGPS()
tt  = TimeScalesFactory.getTT()
tdb = TimeScalesFactory.getTDB()
tcg = TimeScalesFactory.getTCG()
tcb = TimeScalesFactory.getTCB()

times = [utc, tai, gps, tt, tdb, tcg, tcb]
prec = 8

def round_str(txt, prec):
    return str(round(int(txt[:prec+1]),-1))[:prec]

with open("time_converter_orekit.txt", "w") as f:
    for year in [2000, 2025, 2050]:
        date = AbsoluteDate(year, 5, 21, 1, 2, 3.0, utc)
        for time in times:
            line = time.toString().ljust(3) + " "
            line += date.toString(time).ljust(35) + " "
            line += str(date.getMJD(time)).ljust(20) + " "
            line += str(date.offsetFrom(AbsoluteDate.J2000_EPOCH, time)).ljust(10) + "\n"
            f.write(line)
            print(line, end="")

UTC 2000-05-21T01:02:03.000             51685.04309027782    12142987.184
TAI 2000-05-21T01:02:35.000             51685.04346064804    12142987.184
GPS 2000-05-21T01:02:16.000             51685.043240740895   12142987.184
TT  2000-05-21T01:03:07.184             51685.0438331482     12142987.184
TDB 2000-05-21T01:03:07.18513671082713  51685.04383316124    12142987.18520937
TCG 2000-05-21T01:03:07.69829608574647  51685.04383910075    12142987.1924628
TCB 2000-05-21T01:03:18.62713772129806  51685.04396559205    12142987.373488788
UTC 2025-05-21T01:02:03.000             60816.04309027782    801061387.184
TAI 2025-05-21T01:02:40.000             60816.04351851856    801061392.184
GPS 2025-05-21T01:02:21.000             60816.04329861095    801061392.184
TT  2025-05-21T01:03:12.184             60816.04389101826    801061392.184
TDB 2025-05-21T01:03:12.18514673820299  60816.04389103176    801061392.1852194
TCG 2025-05-21T01:03:13.24811621139622  60816.04390333453    801061392.7422829
TCB 2025-

## Coordinate Conversions

In [3]:
from org.hipparchus.geometry.euclidean.threed import Vector3D, SphericalCoordinates
from org.orekit.data import DataProvidersManager, ZipJarCrawler
from org.orekit.frames import FramesFactory, TopocentricFrame, StaticTransform
from org.orekit.bodies import OneAxisEllipsoid, GeodeticPoint, CelestialBodyFactory, PythonCelestialBodies
from org.orekit.time import TimeScalesFactory, AbsoluteDate, DateComponents, TimeComponents
from org.orekit.utils import IERSConventions, Constants, PVCoordinates, PVCoordinatesProvider, AbsolutePVCoordinates

In [4]:
from org.orekit.bodies import JPLEphemeridesLoader

In [5]:
date = AbsoluteDate(2020, 5, 21, 1, 2, 3.0, tai)

In [6]:
gcrf = FramesFactory.getGCRF()
icrf = FramesFactory.getICRF()
itrf = FramesFactory.getITRF(IERSConventions.IERS_2010, True)
moon = CelestialBodyFactory.getMoon()
earth = CelestialBodyFactory.getEarth()
earth_ci = earth.getInertiallyOrientedFrame()
moon_ci = moon.getInertiallyOrientedFrame()
moon_pa = moon.getBodyOrientedFrame()

In [7]:
gcrf.getTransformProvider()

<TransformProvider: org.orekit.frames.FixedTransformProvider@3681037>

In [8]:
earth_ci.getTransformProvider()

<TransformProvider: org.orekit.bodies.JPLCelestialBody$InertiallyOriented$1@2459319c>

In [7]:
rv_zero = PVCoordinates(Vector3D.ZERO, Vector3D.ZERO)

In [59]:
gcrf.getStaticTransformTo(moon_ci, date).getTranslation()

<Vector3D: {-302,155,058.663818; -253,132,209.30597627; -80,187,122.1907977}>

In [18]:
gcrf.getTransformTo(moon_ci, date).transformPVCoordinates(rv_zero)

<PVCoordinates: {P(-2.840314582076087E8, -2.826062963696861E8, 3.554407728680581E7), V(702.9202720583665, -679.5139337145679, -78.92752456278919), A(0.0017525253777095247, 0.0017237045879381291, -2.2110952267113704E-4)}>

In [13]:
rv_mci = gcrf.getTransformTo(moon_ci, date).transformPVCoordinates(rv_gcrf)
rv_mci

<PVCoordinates: {P(2.7976120594803077E8, 2.9144292137973046E8, -4.133310283187623E7), V(5835.939532815328, -2117.922440877919, -2201.0752305351716), A(-0.0017297815578289227, -0.0017718019248468468, 2.93278736692682E-4)}>

In [15]:
rv_itrf = gcrf.getTransformTo(itrf, date).transformPVCoordinates(rv_gcrf)
rv_itrf

<PVCoordinates: {P(501226.1146060517, 3181784.6949694213, 6449738.9917203), V(3341.703432555691, 5921.772487196972, -3180.10652907819), A(0.8663093497658235, -0.470442613329044, -9.96946009432777E-7)}>